# BPMN Process Model Discovery with Inductive Miner

This notebook uses pm4py to discover BPMN process models from role-specific event logs using the inductive miner algorithm.
The discovered models are saved as both .bpmn files and .png images.

In [ ]:
import pm4py
import os
from pathlib import Path
from pm4py.objects.ocel.importer.jsonocel import importer as ocel_importer
from pm4py.objects.ocel.importer.jsonocel.importer import Variants as JsonOcelVariants
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.bpmn.exporter import exporter as bpmn_exporter
from pm4py.visualization.bpmn import visualizer as bpmn_visualizer
from pm4py.objects.ocel.obj import OCEL

## Configuration

In [ ]:
# Define paths
role_logs_dir = Path('role_logs')
output_dir = Path('bpmn_models')
output_dir.mkdir(parents=True, exist_ok=True)

# Define all role names based on the OCEL files
roles = [
    'bot',
    'contributor',
    'devops_engineer',
    'feature_developer',
    'issue_reporter',
    'maintainer',
    'quality_engineer',
    'technical_writer'
]

print(f"Processing {len(roles)} roles")
print(f"Input directory: {role_logs_dir}")
print(f"Output directory: {output_dir}")

## Helper Functions

In [ ]:
def discover_bpmn_for_role(role_name):
    """
    Discover BPMN process model for a specific role using inductive miner.
    
    Args:
        role_name: Name of the role to process
    
    Returns:
        tuple: (bpmn_model, success_flag)
    """
    try:
        # Define file paths
        ocel_file = role_logs_dir / f'ocel_{role_name}.json'
        bpmn_file = output_dir / f'bpmn_{role_name}.bpmn'
        png_file = output_dir / f'bpmn_{role_name}.png'
        
        print(f"\n{'='*60}")
        print(f"Processing role: {role_name}")
        print(f"{'='*60}")
        
        # Load OCEL log
        print(f"Loading OCEL log from: {ocel_file}")
        ocel = ocel_importer.apply(
            str(ocel_file),
            variant=JsonOcelVariants.OCEL20_STANDARD
        )
        
        # Convert OCEL to traditional event log (flattening)
        print("Converting OCEL to traditional event log...")
        object_types = sorted(pm4py.ocel_get_object_types(ocel))
        if not object_types:
            raise ValueError("OCEL contains no object types to flatten on")
        print(f"Available object types: {object_types}")
        chosen_object_type = object_types[0]
        print(f"Flattening on object type: {chosen_object_type}")
        flat_df = pm4py.ocel_flattening(ocel, chosen_object_type)
        unique_activities = flat_df['concept:name'].nunique() if 'concept:name' in flat_df.columns else 0
        case_col = 'case:concept:name'
        avg_events_per_case = flat_df.groupby(case_col).size().mean() if case_col in flat_df.columns and len(flat_df) > 0 else 0
        top_activities = flat_df['concept:name'].value_counts().head(10).to_dict() if 'concept:name' in flat_df.columns else {}
        print(f"Flattened rows: {len(flat_df)}")
        print(f"Unique activities: {unique_activities}")
        print(f"Average events per case: {avg_events_per_case:.2f}")
        print(f"Top activities: {top_activities}")
        event_log = pm4py.convert_to_event_log(flat_df)
        
        print(f"Event log contains {len(event_log)} traces")
        
        # Discover process tree using inductive miner, then convert to BPMN
        print("Discovering process tree with inductive miner...")
        process_tree = inductive_miner.apply(event_log)
        bpmn_model = pm4py.convert_to_bpmn(process_tree)
        
        # Export BPMN model to file
        print(f"Exporting BPMN model to: {bpmn_file}")
        bpmn_exporter.apply(bpmn_model, str(bpmn_file))
        
        # Visualize and save as PNG
        print(f"Generating visualization: {png_file}")
        gviz = bpmn_visualizer.apply(bpmn_model)
        bpmn_visualizer.save(gviz, str(png_file))
        
        print(f"✓ Successfully processed {role_name}")
        return bpmn_model, True
        
    except Exception as e:
        print(f"✗ Error processing {role_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, False

## Process All Roles

In [ ]:
# Process each role and collect results
results = {}
successful = []
failed = []

for role in roles:
    bpmn_model, success = discover_bpmn_for_role(role)
    results[role] = bpmn_model
    
    if success:
        successful.append(role)
    else:
        failed.append(role)

# Print summary
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total roles processed: {len(roles)}")
print(f"Successful: {len(successful)}")
print(f"Failed: {len(failed)}")

if successful:
    print(f"\n✓ Successfully processed roles:")
    for role in successful:
        print(f"  - {role}")

if failed:
    print(f"\n✗ Failed roles:")
    for role in failed:
        print(f"  - {role}")

## Verify Output Files

In [ ]:
# List all generated BPMN files
bpmn_files = sorted(output_dir.glob('bpmn_*.bpmn'))
png_files = sorted(output_dir.glob('bpmn_*.png'))

print(f"\nGenerated BPMN files ({len(bpmn_files)}):")
for f in bpmn_files:
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name} ({size_kb:.2f} KB)")

print(f"\nGenerated PNG files ({len(png_files)}):")
for f in png_files:
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name} ({size_kb:.2f} KB)")

## Display Sample BPMN Model

Display one of the discovered BPMN models as an example.

In [ ]:
# Display the first successful BPMN model
if successful:
    sample_role = successful[0]
    print(f"Displaying BPMN model for: {sample_role}")
    
    if results[sample_role] is not None:
        gviz = bpmn_visualizer.apply(results[sample_role])
        bpmn_visualizer.view(gviz)
else:
    print("No successful models to display")

## Model Statistics

In [ ]:
# Collect statistics about the discovered models
print("\nBPMN Model Statistics:")
print(f"{'Role':<25} {'Nodes':<10} {'Edges':<10}")
print("-" * 45)

for role in successful:
    if results[role] is not None:
        bpmn = results[role]
        num_nodes = len(bpmn.get_nodes())
        num_flows = len(bpmn.get_flows())
        print(f"{role:<25} {num_nodes:<10} {num_flows:<10}")